# Training with cleaned dataset

-------------------------------------------

### Initializing the work environment

```bash
python -m venv venv
source venv/bin/activate
pip install -r requirement.txt
```

### Import the necessary libs

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

--------------------------------------

## Init the dataset

In [ ]:
# init the dataset with Pandas
def init(path):
    df = pd.read_csv(path, header=None, names=[0, 1, 2])
    print(f"dataset load perfectly : {df.shape}")
    return df

df = init("./cleaned_dataset.csv")

-----------------------------------------

## Dataset analysis

This function will show us the most used words, and afterward, we can analyze the results to know what to do with our dataset.

In [ ]:
def most_writen_words(df, num_words=40):
    word_array = df[2].str.split()
    all_words = word_array.explode()
    frequences = all_words.value_counts().head(num_words)

    plt.figure(figsize=(12, 6))
    plt.bar(frequences.index, frequences.values, color='skyblue') 
    plt.title(f'Top {num_words} of the most words in the dataset')
    plt.xlabel('words')
    plt.ylabel('frequences')
    plt.xticks(rotation=45)
    plt.show()

most_writen_words(df)

We can see that the most written words are useless for the training because the model wants words with a real meaning, like a location or a feeling. We don't need words like 'I', 'my', 'and', etc.

-----------------------------------------------

## Words analysis

Fortunately, Scikit-learn provides a function that removes all the useless words (stop words) and converts the text into numerical values, allowing the model to understand what each line is about.

In [ ]:
def vectorize_text(df):
    # plus le mots apparet plus son score seras petit et a l'inverse plus le mot est rare et specifique plus son score est elevée
    vectorizer = TfidfVectorizer(stop_words='english')
    # fit_trasnform vas transformer le "dictionnaire" medical qu'il se creer lui meme en ligne de valeur en fonction des mot (en gros cela creer une matrice dans laquelle les colonnes corresponde a une valeure qui elle meme correspond a un mots plus ou moins important)
    matrice_tfidf = vectorizer.fit_transform(df[2])
    # Pure debug car cela nous donne le "dictionnaire" de mots important
    mots_gardes = vectorizer.get_feature_names_out()
    print(f"Number of medical words kept : {len(mots_gardes)}")

    return matrice_tfidf, vectorizer

X, vectoriseur = vectorize_text(df)

Okay, now we are going to show the useful words kept by the function and try to understand how to find the different clusters.

Exemple extremement simplifier de la matrice X :

|  | Col n0 : "ache" | Col n1 : "chest" |
|:-------- |:--------:| --------:|
| Patient 1| 0.0   | 0.2    |
| Patient 2| 0.4   | 0.1    |

In [ ]:
# X : matrice, vectorizer : Dictionnaire
def plot_top_tfidf_words(X, vectoriseur, top_n=20):

    # cette ligne permet l'addition de toute les valeurs d'une colonne et ce pour toute les colonnes
    somme_tfidf = np.array(X.sum(axis=0)).flatten()
    # cette ligne permet de recuperer les noms en claire pour les differentes colonnes
    mots = vectoriseur.get_feature_names_out()
    # permet de classer les index des 20 mots qui ont le score le plus gros du plus petit au plus grand
    indices_top = somme_tfidf.argsort()[-top_n:]
    # permette d'avoir les mot et les score grace a la liste des indices
    mots_top = [mots[i] for i in indices_top]
    scores_top = somme_tfidf[indices_top]
    
    plt.figure(figsize=(10, 8))
    plt.barh(mots_top, scores_top, color='coral')
    plt.title(f"Top {top_n} of the most 'discriminants' words")
    plt.xlabel("Score")
    plt.ylabel("Medicals words")
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.show()

plot_top_tfidf_words(X, vectoriseur)

Memo english word to french :
- sore = Endolori (qui fait mal)
- ache = Douleur
- cough = Toux
- sharp = Aigu / Vif

Interpretation of the "most discriminative words":

We have some body parts, which will be very useful to give a diagnosis. The body parts we have are: chest, skin, ear, and throat.
We also have some feelings (symptoms) about them: pain (ache), cold, sharp, long, sore, and cough (cough is not a feeling, but a good clue to give a diagnosis).
So, we can now try to understand and think about a way to train an AI to give a diagnosis based on a testimony. For example: if the testimony has "cough" and "throat" in it, it could be tonsillitis (maux de gorge).

---------------------------------

## Cloud Analysis and cluster reflection

Now, we will see how to group the different lines of our dataset into clusters and determine the optimal number of clusters we need.

In [ ]:
# X : matrice
def plot_patients_2d(X):
    # Vas permettre "d'ecraser" nos donner pour qu'elles puissent etre afficher sur un graphique 2d (J'ai choisit Truncated car notre matrice contient bcp de 0)
    svd = TruncatedSVD(n_components=2, random_state=42)
    # "ecrase" les donner pour de bon en faisant deux colonnes le plus pertinent possible
    X_2d = svd.fit_transform(X)

    plt.figure(figsize=(10, 8))
    # vas prendre toute les coords pour l'axe Y et toute les coords pour l'axe X afin de placer les points sur le graph
    plt.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.6, c='dodgerblue', edgecolors='w', s=50)
    plt.title("Visualisation 2D des similarités entre les témoignages")
    plt.xlabel("Composante Mathématique Principale 1")
    plt.ylabel("Composante Mathématique Principale 2")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()
    return X_2d

X_2d = plot_patients_2d(X)

Here, we reduced the dimensionality of our data to 2 parameters (2D) to plot it on a graph. However, as we can see, it is hard to tell which points belong together. To fix this, we will color the points into different clusters based on their similarities.

In [ ]:
def apply_kmeans_and_plot(X, X_2d, df, n_clusters=5):
    # J'ai choisis K-means comme program de machine learning car c'est le plus performant pour l'apprentissage non supervisé et le clustering (il suffit de taper sur le web "best algorithm for non supervised learning and clustering" et c'est le premier resultat)
    # Cette ligne est cruciale car c'est le, moment ou l'on appel l'intelligence artificielle en lui donnant un grainne d'aleatoire precise pour pas que nos resultats change a chaque fois que l'on relance le programme 
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    # Cette ligne permet a notre program de machine learning d'apprendre avec toute la matrice de mots et les valeurs par patients afin de trouver les similitudes 
    kmeans.fit(X)
    # Apres sont apprentissage le program vas disposer les label sur chaque patient afin de creer les clusters
    labels_predict = kmeans.labels_
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    # Attention la représentation shematique en 2D n'est pas dutoput precise au niveau du placement des point (la multi dimmensionalité des données ne permet pas une visualisation aisée des resultats)
    plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels_predict, cmap='viridis', alpha=0.7, edgecolors='w')
    plt.title("VOS Clusters (K-Means)")
    plt.xlabel("Axe 1")
    plt.ylabel("Axe 2")
    
    plt.tight_layout()
    plt.show()

apply_kmeans_and_plot(X, X_2d, df, n_clusters=5) # for now we don't really know how many clusters we have to put in.

As we can observe, points of the same color are mostly grouped in the same area, forming distinct clusters. However, the yellow points are widely spread out. This indicates we need a better way to separate the data into clusters. To achieve this, highly efficient techniques exist, such as the Elbow Method.

--------------------------------

The Elbow Method helps us find the perfect number of clusters. We test the algorithm with 1, 2, 3... up to 12 clusters and calculate the "inertia" (which measures how tight our groups are). Then, we plot these values on a graph.

The curve looks like an arm. The best number of clusters is exactly at the "elbow", where the line stops dropping fast and becomes almost flat. Adding more clusters after this point doesn't improve the result.

In [ ]:
# max_k=10 est une valeur par defaut que j'ai mis car il etait tres probable que le nombre ideal de cluster soit compris entre 1 et 10 (et c'est bien le cas)
def find_optimal_k(X, max_k=10):
    # cette variable vas stocker le score dnner par cluster pour les restituer sur graphique
    inertias = []

    K_range = range(1, max_k + 1)
    
    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
        kmeans.fit(X)
        # cela vas ajouter le score d'inertie (l'inertie est globalement le calcule de la distance entre tout les points d'un cluster et son centre ce qui fais que plus le chifre est bas mieux c'est)
        inertias.append(kmeans.inertia_)
    
    plt.figure(figsize=(10, 6))
    plt.plot(K_range, inertias, marker='o', linestyle='-', color='teal', markersize=8)
    plt.title("Méthode du Coude pour trouver le K optimal")
    plt.xlabel("Nombre de clusters (K)")
    plt.ylabel("Inertie (Distance intra-cluster)")
    plt.xticks(K_range) # Pour bien afficher tous les nombres de 1 à 10
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

find_optimal_k(X, max_k=12)

To interpret the Elbow Method graph, we must identify a significant change in the slope of the curve. Here, we can observe a very clear 'elbow' at K=9 (before the line becomes flat at 10), which gives us the optimal number of clusters for our dataset. It is worth pointing out that this dataset was clearly custom-made for this exercise: the shape of the curve is literally perfect, making the ideal number of clusters incredibly obvious to discover.

As we can observe, we do not simply select the lowest possible inertia. We must find a balance between a low inertia and a reasonable number of clusters. For instance, creating 1,000 clusters for 1,000 patients would result in an inertia of 0, but this would lead to overfitting and defeat the whole purpose of clustering.

we can now retry with 9 cluster :

In [ ]:
apply_kmeans_and_plot(X, X_2d, df, n_clusters=9)

This choice is much more pertinent. Even if some points appear very far from the others, we must remember that we reduced many dimensions (parameters) to project these points onto a 2D graph, which distorts the real distances.